# Philippine government sentiment study — 2026

**Group:** replace with your actual group name before submission.

This notebook runs the reproducible CSV pipeline for public discussion about the Philippine national administration during 2026. It does not contain invented posts, manufactured human labels, or prewritten conclusions.

Use `docs/methodology.md` for the study design and `README.md` for installation and API access. The attached *Midterm Activity 1* supplies assignment requirements; this notebook focuses on the requested code and scripts. Save a submission copy with the instructor's required `GROUP NAME.ipynb` filename.

**Interpretation boundary:** sentiment in the collected posts is not a population approval estimate or an objective measurement of government performance. Performance-related topics identify discussion; claims about actual performance need separate evidence.

## 1. Setup and reproducibility

Install the project in the Python environment used by this notebook, using the README instructions. Keep API tokens in environment variables or the project's supported environment file. Do not paste credentials into notebook cells.

The helpers below invoke the CLI with an argument list, preview CSVs using escaped HTML, and locate the repository from either its root or the notebook directory. Generated datasets remain CSV files. With no real inputs, later stages explain what is missing and skip computation.

In [ ]:
import csv
import html
import subprocess
import sys
from collections import Counter
from pathlib import Path

from IPython.display import HTML, display


def find_repository() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError(
        "Open this notebook from the repository root or its notebooks directory."
    )


REPO = find_repository()
CONFIG = REPO / "configs" / "research.example.toml"
RAW_DIR = REPO / "data" / "raw"
PROCESSED = REPO / "data" / "processed" / "posts.csv"
AUDIT = REPO / "data" / "processed" / "preprocessing_audit.csv"
ANALYSIS_DIR = REPO / "data" / "analysis"
ANNOTATIONS = REPO / "data" / "labels" / "annotation.csv"
EVALUATION_DIR = REPO / "data" / "evaluation"

# Set True only after reviewing which derived outputs will be replaced.
# Existing raw collections and human annotations are always preserved here.
FORCE_REBUILD = False


def run_cli(*arguments: str) -> None:
    command = [sys.executable, "-m", "ph_sentiment", *map(str, arguments)]
    result = subprocess.run(command, cwd=REPO, capture_output=True, text=True, check=False)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        detail = result.stderr.strip() or "No diagnostic was returned."
        raise RuntimeError(f"Pipeline command failed ({result.returncode}): {detail}")
    if result.stderr.strip():
        print(result.stderr.strip())


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def preview_csv(path: Path, limit: int = 6, max_columns: int = 8) -> None:
    if not path.exists():
        print(f"Not generated: {path.relative_to(REPO)}")
        return
    rows = read_csv(path)
    print(f"{path.relative_to(REPO)}: {len(rows):,} rows")
    if not rows:
        print("No rows to preview.")
        return

    columns = list(rows[0])[:max_columns]
    headings = "".join(f'<th scope="col">{html.escape(name)}</th>' for name in columns)
    body = []
    for row in rows[:limit]:
        values = []
        for name in columns:
            value = str(row.get(name, ""))
            if len(value) > 180:
                value = value[:180] + "…"
            values.append(f"<td>{html.escape(value)}</td>")
        body.append("<tr>" + "".join(values) + "</tr>")
    caption = html.escape(
        f"{path.name}: first {min(limit, len(rows))} rows; first {len(columns)} columns"
    )
    display(
        HTML(
            f"<table><caption>{caption}</caption><thead><tr>{headings}</tr></thead>"
            f"<tbody>{''.join(body)}</tbody></table>"
        )
    )


def overwrite_arguments() -> list[str]:
    return ["--overwrite"] if FORCE_REBUILD else []


print(f"Repository: {REPO}")
print(f"Python: {sys.version.split()[0]}")
print(f"Interpreter: {sys.executable}")
print(f"Research configuration: {CONFIG.relative_to(REPO)}")

## 2. Define the scope and collect public records

Review the example research configuration before collection. Use an inclusive mix of verified administration aliases, institutions, and policy terms in English, Filipino, and Taglish. Document the exact queries and collection time. The study period starts on **2026-01-01** and ends before **2027-01-01**; collections before year-end cover only the elapsed period.

For the recommended one-command alternative, run `python -m ph_sentiment collect-live --config configs/research.example.toml` after filling the local `.env` placeholders. It attempts all four platforms, reports missing credentials and source failures in `collection_report.csv`, preserves partial results, and automatically prepares and analyzes once the configured minimum is met (200 by default). Repeat `--platform` to select sources or add `--skip-analysis` to collect and prepare only. A new timestamped `data/live` folder contains raw/processed CSVs, optional analysis CSVs, and `run_summary.csv`; missing credentials produce status reports, not fabricated observations.

The separate-stage cell below remains disabled until you deliberately enable it. Its later cells use the setup paths under `data/raw`, `data/processed`, and `data/analysis`. To review a one-command live run here, update those paths to that run's folders first. Official API permissions and history limits differ by platform. Inspect source diagnostics and report actual coverage. Imported data must use the README's CSV schema and authorized public sources.

The final prepared dataset must contain **at least 200 cleaned records from different sources**. A pilot corpus can exercise the pipeline, but it does not meet the final count requirement.

In [ ]:
# Enable only after reviewing the configuration and setting source credentials.
RUN_COLLECTION = False
PLATFORMS = ["reddit"]  # Add supported sources for which you have API access.

if RUN_COLLECTION:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for platform in PLATFORMS:
        destination = RAW_DIR / f"{platform}.csv"
        if destination.exists():
            print(f"Preserving existing raw collection: {destination.name}")
            continue
        try:
            run_cli(
                "collect",
                "--platform",
                platform,
                "--config",
                str(CONFIG),
                "--output",
                str(destination),
            )
        except RuntimeError as error:
            # Keep other source attempts usable; do not substitute generated posts.
            print(f"{platform}: {error}")
else:
    print("Collection disabled. Supply real raw CSVs or explicitly enable collection.")

## 3. Prepare and audit the CSV corpus

The preparation stage applies configured date rules, normalization, and duplicate handling. Raw files remain separate. Search terms guide collection; preparation does not establish political relevance automatically. Review the audit, retained samples, and imported records for relevance.

By default, the next cell uses dataset CSVs in `data/raw`, excluding collection manifest CSVs. Set `RAW_FILES` to an explicit list for a controlled run. Keep annotation and analysis files outside that directory. Set `FORCE_REBUILD = True` in the setup cell only when you intend to replace derived outputs after changing the inputs or configuration.

In [ ]:
RAW_FILES = (
    sorted(path for path in RAW_DIR.glob("*.csv") if not path.stem.endswith("_collection_manifest"))
    if RAW_DIR.exists()
    else []
)
# Example explicit selection:
# RAW_FILES = [RAW_DIR / "reddit.csv", RAW_DIR / "x.csv"]

if not RAW_FILES:
    print("Preparation skipped: no real CSV inputs in data/raw.")
elif PROCESSED.exists() and not FORCE_REBUILD:
    print("Using existing prepared CSV. Review its provenance before reuse.")
else:
    arguments = ["prepare", "--config", str(CONFIG)]
    for source in RAW_FILES:
        arguments.extend(["--input", str(source)])
    arguments.extend(
        [
            "--output",
            str(PROCESSED),
            "--audit-output",
            str(AUDIT),
            *overwrite_arguments(),
        ]
    )
    run_cli(*arguments)

preview_csv(AUDIT)
preview_csv(PROCESSED)

In [ ]:
if PROCESSED.exists():
    prepared_rows = read_csv(PROCESSED)
    platform_counts = Counter(row.get("platform", "") for row in prepared_rows)
    print(f"Final prepared rows: {len(prepared_rows):,}")
    print(f"Actual source counts: {dict(sorted(platform_counts.items()))}")
    populated_sources = [name for name, count in platform_counts.items() if name and count]
    if len(prepared_rows) < 200 or len(populated_sources) < 2:
        print(
            "Pilot only: the activity requires at least 200 cleaned records from different sources."
        )
    else:
        print(
            "Count/source checks passed. Confirm relevance, time coverage, "
            "and independence by inspecting the data and audit."
        )
else:
    prepared_rows = []
    print("Quality checks skipped: prepare the real corpus first.")

### Record the quality review

Complete these notes after inspecting real data:

- Collection time, query vocabulary, endpoint coverage, and observed date interval:
- Retained counts by platform and reasons for exclusions:
- Relevance checks, multilingual coverage, and suspected near-duplicate content:
- Remaining sampling limitations:

Do not interpret failed or empty source requests as neutral sentiment.

## 4. Fit three LDA topics and compute rule sentiment

The analyzer defaults to three LDA topics with a fixed seed. LDA uses word counts; document proximity uses normalized TF-IDF and token sets. The fixed lexical baseline predicts positive, negative, or neutral sentiment with English/Filipino/Taglish rules.

The sentiment score is heuristic, not confidence. Rule matches can represent general emotional tone rather than stance toward the administration. Use independent human labels to measure that error. Keep record types separate: Facebook Page posts can be institutional communications, while comments can express public reactions. The summary groups record types separately and uses Manila-local months.

In [ ]:
analysis_marker = ANALYSIS_DIR / "analyzed_documents.csv"

if not prepared_rows:
    print("Analysis skipped: no retained prepared records are available.")
elif analysis_marker.exists() and not FORCE_REBUILD:
    print("Using existing analysis CSVs. Rebuild if the prepared corpus changed.")
else:
    run_cli(
        "analyze",
        "--config",
        str(CONFIG),
        "--input",
        str(PROCESSED),
        "--output-dir",
        str(ANALYSIS_DIR),
        *overwrite_arguments(),
    )

for filename in (
    "analyzed_documents.csv",
    "topic_terms.csv",
    "topic_metrics.csv",
    "sentiment_summary.csv",
):
    preview_csv(ANALYSIS_DIR / filename)

### Topic interpretation and topic-model evaluation

Name each topic only after reading its top terms and representative posts. Check overlapping terms, poorly assigned documents, and whether three topics are actually interpretable.

`topic_metrics.csv` records document-presence NPMI coherence, top-term diversity, and explicitly **in-sample** perplexity. These are descriptive diagnostics; they do not establish topic validity or predictive generalization.

| Topic | Name supported by records | Representative record IDs | Interpretation and limitations |
| --- | --- | --- | --- |
| 0 | Complete after inspection | Complete after inspection | Complete after inspection |
| 1 | Complete after inspection | Complete after inspection | Complete after inspection |
| 2 | Complete after inspection | Complete after inspection | Complete after inspection |

Report values from the generated topic metrics, together with the human review. Never replace missing topic output with assumed results.

## 5. Examine cosine, Jaccard, and Euclidean proximity

Larger cosine and Jaccard similarity mean more similar wording under their respective representations. Smaller Euclidean distance means closer normalized TF-IDF vectors. Similar wording does not establish agreement, accuracy, shared authorship, or coordinated behavior.

The pair limit bounds computation, so the CSV may contain only selected pairs. Inspect `pair_status` and exclude empty-representation results from substantive interpretation.

In [ ]:
preview_csv(ANALYSIS_DIR / "proximity_pairs.csv", limit=10)

### Proximity interpretation

Select actual pairs from the CSV and inspect both underlying posts:

- Pair IDs and the three measured values:
- Shared and differing wording that explains the comparison:
- Whether text similarity corresponds to stance similarity in this example:
- Pair-status and sampling limitations:

Keep examples traceable to record IDs. Do not quote unnecessary personal information in the submission.

## 6. Create independent human sentiment labels

The label command creates a CSV template with a blank `gold_label` column and no model predictions. Reviewers should work without seeing rule sentiment outputs. Existing annotation files are always preserved by this notebook.

Define the annotation target as sentiment toward the Philippine national administration or its governance:

- **positive:** clear approval or praise of the target.
- **negative:** clear criticism or disapproval of the target.
- **neutral:** discussion or information without a clear stance toward the target.

Sarcasm, quoted claims, mixed stance, and ambiguous targets need review or adjudication. Do not label unresolved cases neutral just to fill a row. If possible, have a second reviewer independently label a subset and report actual agreement. Never copy rule predictions into `gold_label`.

If the prepared corpus changes, generate a new annotation file explicitly and reconcile IDs; do not erase completed human work.

In [ ]:
if ANNOTATIONS.exists():
    print(f"Preserving human annotation file: {ANNOTATIONS.relative_to(REPO)}")
elif not prepared_rows:
    print("Annotation template skipped: prepare real data first.")
else:
    run_cli(
        "label",
        "--config",
        str(CONFIG),
        "--input",
        str(PROCESSED),
        "--output",
        str(ANNOTATIONS),
    )

if ANNOTATIONS.exists():
    annotation_rows = read_csv(ANNOTATIONS)
    label_counts = Counter(row.get("gold_label", "").strip() for row in annotation_rows)
    print(f"Human-label counts, including blanks: {dict(sorted(label_counts.items()))}")
    print("Complete and adjudicate the gold_label column before final evaluation.")
else:
    annotation_rows = []

## 7. Evaluate: stratified 90/10 holdout and 10-fold development CV

Only independent human labels count as ground truth. The evaluator creates a stratified 90% development / 10% holdout split, then uses ten stratified folds within development. All three classes must be present, and each development class must support ten folds. If a class is too small, obtain and annotate more relevant data instead of changing valid labels.

The rule model is fixed and has no fitting step. Its cross-validation scores describe variation across development subsets. They do not imply learning on training folds. Keep holdout results separate from development-fold metrics.

Inspect development errors before a rule revision. After inspecting holdout errors to change the rules, obtain a fresh untouched holdout for a final assessment.

In [ ]:
# Deliberately enable only after independent annotation and adjudication.
RUN_EVALUATION = False
evaluation_marker = EVALUATION_DIR / "evaluation_metrics.csv"

if not RUN_EVALUATION:
    print("Evaluation disabled until independent human annotation is complete.")
elif not annotation_rows:
    print("Evaluation skipped: no human annotation CSV is available.")
elif evaluation_marker.exists() and not FORCE_REBUILD:
    print("Using existing evaluation CSVs. Rebuild only for a documented new run.")
else:
    run_cli(
        "evaluate",
        "--config",
        str(CONFIG),
        "--input",
        str(ANNOTATIONS),
        "--output-dir",
        str(EVALUATION_DIR),
        *overwrite_arguments(),
    )

for filename in (
    "evaluation_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "split_assignments.csv",
    "evaluation_predictions.csv",
):
    preview_csv(EVALUATION_DIR / filename)

### Evaluation interpretation

Complete these notes from the actual CSVs:

- Number of independently labeled posts, class balance, annotation process, and adjudication:
- Development-fold mean/variation and holdout accuracy, macro precision, recall, F1, and balanced accuracy:
- Per-class confusion and representative development errors:
- Model version, split reproducibility, and remaining language/target limitations:

Use scope fields to distinguish holdout results from development-fold results. A high score against labels copied from the model would be circular and invalid.

## 8. Scripts, outputs, and remarks

All tabular datasets generated by this workflow are CSV files.

| Script / command | Output | Remarks |
| --- | --- | --- |
| `ph_sentiment collect-live` | A timestamped live run's raw/processed/analysis CSVs and status reports | Recommended automated workflow; missing credentials and partial-source results are explicit. |
| `ph_sentiment collect` | `data/raw/<platform>.csv` | Real public records returned by configured supported APIs. |
| `ph_sentiment prepare` | `data/processed/posts.csv`, `preprocessing_audit.csv` | Final cleaned corpus and audit; check at least 200 records and different sources. |
| `ph_sentiment analyze` | `analyzed_documents.csv`, `topic_terms.csv`, `topic_metrics.csv` | Three requested LDA topics, fixed-rule predictions, and topic diagnostics. |
| Same analysis command | `proximity_pairs.csv`, `sentiment_summary.csv` | Three proximity measures and descriptive sentiment counts/shares. |
| `ph_sentiment label` | `data/labels/annotation.csv` | Blank human ground-truth template. |
| `ph_sentiment evaluate` | Metrics, reports, confusion matrices, split assignments, predictions | Stratified holdout and ten development folds; all outputs CSV. |
| This notebook | Saved executed cells, tables, and reviewed interpretation | Rename a submission copy to `GROUP NAME.ipynb`. |

Summarize sentiment by topic and source only within the observed corpus. Platform users self-select; APIs restrict coverage; duplicated content, bots, language, sarcasm, and target errors can distort results. Corpus sentiment alone does not measure actual government performance. Record these limits alongside any conclusion.